# Inside vs Outside Wind (Dome Closed + AHU Off)

This notebook plots inside wind speed from ESS anemometers against outside wind speed,
filtered to times when:
- the dome aperture shutters are closed
- HVAC AHUs are off

It follows the style of `ESS_Wind_vs_DomePointing.ipynb` with updated guards.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.time import Time, TimeDelta
import astropy.units as u

from lsst_efd_client import EfdClient

%matplotlib inline

In [ ]:
client = EfdClient("usdf_efd")
print("Connected to EFD")

## Configuration

In [ ]:
# Time range
t_end = Time("2026-03-15T05:00:00", scale="utc")
t_start = t_end - TimeDelta(4 * u.week)

print(f"Start: {t_start.iso}")
print(f"End  : {t_end.iso}")

# Wind topics/fields
inside_topic = "lsst.sal.ESS.airTurbulence"
outside_topic = "lsst.sal.ESS.airFlow"
inside_speed_field = "speedMagnitude"
outside_speed_candidates = ["speedMagnitude", "speed"]
outside_index = 301

inside_sensors = {
    110: "TMA Platform",
    123: "Top Ring -X/-Y",
    124: "Top Ring +X/-Y",
    125: "Top Ring +X/+Y",
    126: "Top Ring -X/+Y",
}

# Dome-closed guard
dome_topic = "lsst.sal.MTDome.apertureShutter"
shutter_field_a = "positionActual0"
shutter_field_b = "positionActual1"
closed_threshold_pct = 5.0  # both shutters <= threshold means closed

# AHU-off guard (provided mapping)
AHU_TOPICS = [
    "lsst.sal.HVAC.lowerAHU01P05",
    "lsst.sal.HVAC.lowerAHU02P05",
    "lsst.sal.HVAC.lowerAHU03P05",
    "lsst.sal.HVAC.lowerAHU04P05",
]
AHU_STATE_FIELD = "workingState"
AHU_REQUIRED_ON = False  # Explicit guard: require AHUs OFF.

# Require all AHUs off (True) or any AHU off (False).
require_all_ahus_off = True

# Spurious inside-speed clip
inside_speed_max_ms = 15.0  # m/s — removes anemometer artefacts above this value

# Louver-closed guard
louver_topic = "lsst.sal.MTDome.louvers"
louver_pos_field_prefix = (
    "positionActual"  # fields: positionActual0 .. positionActual33
)
louver_indices = [2, 11, 12, 20, 21, 29]  # only enabled/working louvers
louver_closed_threshold_pct = (
    0  # max(positionActual) <= 0: 0=closed, 100=open, -1=disabled
)

# ── Parquet cache ────────────────────────────────────────────────────
CACHE_FILE = "../data/ess_inside_outside_wind.parquet"
FORCE_REFETCH = False  # True → ignore cache, re-fetch everything
INCREMENTAL_REFETCH = False  # True → extend cache with newer data

In [ ]:
import pathlib as _pl

_cp = _pl.Path(CACHE_FILE)
_from_cache = False
if not FORCE_REFETCH and _cp.exists():
    df_work = pd.read_parquet(_cp)
    base = df_work.copy()
    _from_cache = True
    print(f"✓ Loaded {len(df_work):,} rows from cache: {_cp}")
elif FORCE_REFETCH:
    print("FORCE_REFETCH=True — bypassing cache")
else:
    print(f"No cache at {_cp} — will fetch fresh data")

## Optional: Discover HVAC/AHU Topics

In [ ]:
all_topics = await client.get_topics()
hvac_like = sorted(
    [
        t
        for t in all_topics
        if any(
            k in t.lower()
            for k in [
                "hvac",
                "ahu",
                "airhandler",
                "air_handler",
                "airsupply",
                "air_supply",
            ]
        )
    ]
)
print(f"Found {len(hvac_like)} HVAC-like topics")
for t in hvac_like:
    print(t)

In [ ]:
# Inspect schemas for configured AHU topics.
for t in AHU_TOPICS:
    print(f"=== {t} ===")
    schema = await client.get_schema(t)
    display(schema)

## Fetch Wind + Dome Data

In [ ]:
if not _from_cache:
    inside_frames = []
    for idx, label in inside_sensors.items():
        df = await client.select_time_series(
            inside_topic,
            fields=[inside_speed_field],
            start=t_start,
            end=t_end,
            index=idx,
        )
        if df.empty:
            print(f"Inside sensor {idx} ({label}): no data")
            continue
        df = (
            df[[inside_speed_field]]
            .rename(columns={inside_speed_field: "inside_speed"})
            .copy()
        )
        df["sensor_index"] = idx
        df["sensor_label"] = label
        inside_frames.append(df)
        print(f"Inside sensor {idx} ({label}): {len(df)} rows")

    if not inside_frames:
        raise RuntimeError("No inside wind data found in selected time range.")

    inside_all = pd.concat(inside_frames).sort_index()
    inside_mean = (
        inside_all.groupby(level=0)["inside_speed"]
        .median()
        .to_frame(name="inside_speed")
    )

    outside = await client.select_time_series(
        outside_topic,
        fields="*",
        start=t_start,
        end=t_end,
        index=outside_index,
    )
    if outside.empty:
        raise RuntimeError("No outside wind data found.")

    outside_speed_field = next(
        (c for c in outside_speed_candidates if c in outside.columns), None
    )
    if outside_speed_field is None:
        raise RuntimeError(
            f"No outside speed field found. Tried: {outside_speed_candidates}. Available: {list(outside.columns)}"
        )

    outside = outside[[outside_speed_field]].rename(
        columns={outside_speed_field: "outside_speed"}
    )

    dome = await client.select_time_series(
        dome_topic,
        fields=[shutter_field_a, shutter_field_b],
        start=t_start,
        end=t_end,
    )
    if dome.empty:
        raise RuntimeError("No dome apertureShutter data found.")

    louver_fields = [f"{louver_pos_field_prefix}{i}" for i in louver_indices]
    louvers = await client.select_time_series(
        louver_topic,
        fields=louver_fields,
        start=t_start,
        end=t_end,
    )
    if louvers.empty:
        raise RuntimeError("No louver position data found.")

    print(f"inside_mean rows: {len(inside_mean)}")
    print(f"outside rows    : {len(outside)} (field={outside_speed_field})")
    print(f"dome rows       : {len(dome)}")
    print(
        f"louver rows     : {len(louvers)} | example fields: {list(louvers.columns[:4])}"
    )

## AHU-Off Mask Helpers

In [ ]:
def _to_seconds(index):
    # Convert pandas datetime index to float seconds for interpolation.
    return index.view("int64") / 1e9


def _series_matches_required_state(s, required_on=True):
    s_nonnull = s.dropna()
    if s_nonnull.empty:
        return pd.Series(False, index=s.index)

    if pd.api.types.is_bool_dtype(s_nonnull):
        # switchedOn=True means ON.
        return s.fillna(False) if required_on else ~s.fillna(True)

    if pd.api.types.is_numeric_dtype(s_nonnull):
        vals = set(pd.Series(s_nonnull).astype(float).round(6).unique().tolist())
        if vals.issubset({0.0, 1.0}):
            target = 1.0 if required_on else 0.0
            return s.fillna(np.nan).astype(float) == target
        if required_on:
            return s.fillna(-np.inf).astype(float) > 0.0
        return s.fillna(np.inf).astype(float) <= 0.0

    txt = s.astype(str).str.lower()
    on_terms = ["on", "enabled", "running", "true"]
    off_terms = ["off", "disabled", "stopped", "false", "standby"]
    if required_on:
        return txt.str.contains("|".join(on_terms), na=False)
    return txt.str.contains("|".join(off_terms), na=False)


async def fetch_ahu_mask(start, end):
    if not AHU_TOPICS:
        print("AHU_TOPICS is empty: skipping AHU filtering.")
        return None

    per_topic = []
    for topic in AHU_TOPICS:
        try:
            ahu_df = await client.select_time_series(
                topic, fields=[AHU_STATE_FIELD], start=start, end=end
            )
        except Exception as e:
            raise RuntimeError(f"Failed AHU query for topic={topic}: {e}")

        if ahu_df.empty:
            raise RuntimeError(f"AHU query returned no rows for topic={topic}.")

        if AHU_STATE_FIELD not in ahu_df.columns:
            raise RuntimeError(
                f"Field '{AHU_STATE_FIELD}' not found in {topic}. Available columns: {list(ahu_df.columns)}"
            )

        state_match = (
            _series_matches_required_state(
                ahu_df[AHU_STATE_FIELD], required_on=AHU_REQUIRED_ON
            )
            .astype(bool)
            .sort_index()
        )
        per_topic.append(state_match.rename(topic))
        print(f"{topic}: rows={len(ahu_df)}")

    joined = pd.concat(per_topic, axis=1).sort_index()
    if require_all_ahus_off:
        mask = joined.all(axis=1)
    else:
        mask = joined.any(axis=1)

    return mask.rename("ahu_required_state")

## Build Combined Guard and Plot

In [ ]:
if not _from_cache:
    # Align on inside wind timestamps.
    base = inside_mean.copy().sort_index()

    # Outside wind -> nearest in time via interpolation of value on timestamp axis.
    t_base = _to_seconds(base.index)
    t_out = _to_seconds(outside.index)
    base["outside_speed"] = np.interp(
        t_base, t_out, outside["outside_speed"].astype(float).values
    )

    # Dome-closed mask (both shutters <= closed_threshold_pct), interpolated to base timestamps.
    dome_closed_series = (
        (dome[shutter_field_a] <= closed_threshold_pct)
        & (dome[shutter_field_b] <= closed_threshold_pct)
    ).astype(float)
    t_dome = _to_seconds(dome_closed_series.index)
    base["dome_closed"] = np.interp(t_base, t_dome, dome_closed_series.values) >= 0.5

    # Louver-closed mask (all positionActual fields <= threshold), interpolated to base timestamps.
    louver_pos_cols = [f"{louver_pos_field_prefix}{i}" for i in louver_indices]
    louver_closed_series = (
        louvers[louver_pos_cols].max(axis=1) <= louver_closed_threshold_pct
    ).astype(float)
    t_louver = _to_seconds(louver_closed_series.index)
    base["louvers_closed"] = (
        np.interp(t_base, t_louver, louver_closed_series.values) >= 0.5
    )

    # AHU required-state mask if configured.
    ahu_mask = await fetch_ahu_mask(t_start, t_end)
    if ahu_mask is None:
        base["ahu_required_state"] = True
    else:
        t_ahu = _to_seconds(ahu_mask.index)
        base["ahu_required_state"] = (
            np.interp(t_base, t_ahu, ahu_mask.astype(float).values) >= 0.5
        )

    base["good"] = (
        base["dome_closed"] & base["louvers_closed"] & base["ahu_required_state"]
    )


plot_df = base.loc[base["good"], ["inside_speed", "outside_speed"]].dropna()
plot_df = plot_df[plot_df["inside_speed"] <= inside_speed_max_ms]

state_label = "ON" if AHU_REQUIRED_ON else "OFF"
print(f"Total points                 : {len(base)}")
print(f"Dome-closed points           : {int(base['dome_closed'].sum())}")
print(f"Louvers-closed points        : {int(base['louvers_closed'].sum())}")
print(f"AHU-{state_label} points             : {int(base['ahu_required_state'].sum())}")
print(
    f"Combined filtered points     : {len(plot_df)} (after {inside_speed_max_ms} m/s clip)"
)

if len(plot_df) == 0:
    raise RuntimeError(
        "No points after filtering. Check AHU topics/field and thresholds."
    )

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(plot_df["outside_speed"], plot_df["inside_speed"], s=8, alpha=0.25)

xy_max = float(
    np.nanmax([plot_df["outside_speed"].max(), plot_df["inside_speed"].max()])
)
xy_max = max(xy_max, 1.0)
ax.plot([0, xy_max], [0, xy_max], "k--", lw=1, label="y = x")

ax.set_xlim(0, xy_max)
ax.set_ylim(0, xy_max)
ax.set_xlabel("Outside wind speed (m/s)")
ax.set_ylabel("Inside wind speed median (m/s)")
ax.set_title(f"Inside vs Outside Wind | Dome Closed + AHU {state_label}")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ── Save / update parquet cache ────────────────────────────────────────
import pathlib as _pl

_cp = _pl.Path(CACHE_FILE)
if _from_cache:
    print("Loaded from cache — skipping save")
elif INCREMENTAL_REFETCH and _cp.exists():
    _old = pd.read_parquet(_cp)
    _combined = pd.concat([_old, base]).sort_index()
    _combined = _combined[~_combined.index.duplicated(keep="last")]
    _combined.to_parquet(_cp)
    print(f"✓ Appended incremental data → {len(_combined):,} rows total: {_cp}")
else:
    base.to_parquet(_cp)
    print(f"✓ Saved {len(base):,} rows → {_cp}")

In [ ]:
# Optional diagnostics: correlation and robust slope through origin.
x = plot_df["outside_speed"].values
y = plot_df["inside_speed"].values

corr = np.corrcoef(x, y)[0, 1]
slope_origin = np.dot(x, y) / np.dot(x, x) if np.dot(x, x) > 0 else np.nan

print(f"Pearson r          : {corr:.3f}")
print(f"Slope through origin (inside/outside): {slope_origin:.3f}")

## Wind Blocking Efficiency Time Series

Blocking efficiency = 1 - (inside_speed / outside_speed), computed as a
30-minute rolling median over dome-closed samples. Values near 1 mean the dome
is blocking almost all outside wind; values near 0 mean full penetration.

In [ ]:
# Work on dome-closed rows only (all guards: dome + louvers + AHU).
ts = base.loc[base["dome_closed"], ["inside_speed", "outside_speed"]].copy()
ts = ts[ts["inside_speed"] <= inside_speed_max_ms]
ts = ts[
    ts["outside_speed"] > 2.0
]  # minimum outside wind to avoid noise-dominated efficiency

# Penetration ratio and blocking efficiency
ts["penetration"] = (ts["inside_speed"] / ts["outside_speed"]).clip(0, None)
ts["efficiency"] = (1.0 - ts["penetration"]).clip(0, 1)

# 30-minute rolling median (center-aligned)
roll = ts["efficiency"].rolling("30min", center=True, min_periods=10).median()

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# --- Panel 1: outside and inside wind speed ---
ax0 = axes[0]
ax0.plot(
    ts.index, ts["outside_speed"], lw=0.5, alpha=0.6, color="steelblue", label="Outside"
)
ax0.plot(
    ts.index,
    ts["inside_speed"],
    lw=0.5,
    alpha=0.6,
    color="darkorange",
    label="Inside (median of sensors)",
)
ax0.set_ylabel("Wind speed (m/s)")
ax0.legend(loc="upper right", fontsize=8)
ax0.grid(True, alpha=0.3)

# --- Panel 2: raw penetration ratio ---
ax1 = axes[1]
ax1.scatter(ts.index, ts["penetration"], s=1, alpha=0.15, color="gray", label="Sample")
ax1.axhline(1.0, color="k", lw=0.8, ls="--")
ax1.set_ylabel("Penetration\n(inside / outside)")
ax1.set_ylim(0, None)
ax1.grid(True, alpha=0.3)

# --- Panel 3: rolling blocking efficiency ---
ax2 = axes[2]
ax2.plot(
    roll.index,
    roll.values * 100,
    lw=1.2,
    color="seagreen",
    label="30-min rolling median",
)
ax2.axhline(100, color="k", lw=0.8, ls="--", alpha=0.5)
ax2.axhline(0, color="k", lw=0.8, ls="--", alpha=0.5)
ax2.fill_between(roll.index, roll.values * 100, 0, alpha=0.2, color="seagreen")
ax2.set_ylabel("Blocking efficiency (%)")
ax2.set_ylim(0, 105)
ax2.set_xlabel("UTC")
ax2.legend(loc="upper right", fontsize=8)
ax2.grid(True, alpha=0.3)

fig.suptitle(
    f"Dome Wind Blocking Efficiency (dome closed, louvers {louver_indices}, outside > 2 m/s)\n"
    f"{t_start.iso[:10]} – {t_end.iso[:10]}",
    fontsize=11,
)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"Samples used        : {len(ts)}")
print(f"Median efficiency   : {ts['efficiency'].median()*100:.1f}%")
print(f"Mean efficiency     : {ts['efficiency'].mean()*100:.1f}%")
print(f"Std efficiency      : {ts['efficiency'].std()*100:.1f}%")

## Air Leak Rate Through Azimuth Perimeter Gap

Infiltration via the orifice equation:

$$Q = C_d \cdot A_{gap} \cdot \sqrt{\frac{2\,|\Delta P_{dyn}|}{\rho}}$$

- $\Delta P_{dyn}$ = dynamic (wind-driven) pressure difference, obtained by subtracting a 1-hour rolling median from the raw inside−outside differential to remove the static sensor offset.
- $\rho$ = local air density from the outside pressure sensor.
- $C_d$ = 0.6 (sharp-edged gap).
- $A_{gap}$ = π × 30 m × 0.05 m ≈ 4.71 m² (full azimuth perimeter).

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
pressure_topic = "lsst.sal.ESS.pressure"
pressure_field = "pressureItem0"
outside_pres_index = 301  # ESS:301 outside weather station
inside_pres_index = 113  # ESS:113 inside dome

dome_diameter_m = 30.0  # outer dome diameter [m]
gap_height_m = 0.05  # azimuth perimeter gap height [m]
C_d = 0.6  # discharge coefficient (sharp-edged orifice)
R_air = 287.1  # specific gas constant for dry air [J/(kg·K)]
T_assumed_K = 283.0  # ~10 °C fallback if no temperature sensor

# Derived gap area
A_gap = np.pi * dome_diameter_m * gap_height_m
print(f"Gap area: π × {dome_diameter_m} m × {gap_height_m} m = {A_gap:.2f} m²")

# ── Fetch pressure series ────────────────────────────────────────────────────
p_out_raw = await client.select_time_series(
    pressure_topic,
    fields=[pressure_field],
    start=t_start,
    end=t_end,
    index=outside_pres_index,
)
p_in_raw = await client.select_time_series(
    pressure_topic,
    fields=[pressure_field],
    start=t_start,
    end=t_end,
    index=inside_pres_index,
)
if p_out_raw.empty or p_in_raw.empty:
    raise RuntimeError("Pressure data missing for one or both sensors.")

p_out_raw = p_out_raw[[pressure_field]].rename(columns={pressure_field: "p_out"})
p_in_raw = p_in_raw[[pressure_field]].rename(columns={pressure_field: "p_in"})
print(
    f"Outside pressure rows : {len(p_out_raw)}  range {p_out_raw.p_out.min():.0f}–{p_out_raw.p_out.max():.0f} Pa"
)
print(
    f"Inside  pressure rows : {len(p_in_raw)}  range {p_in_raw.p_in.min():.0f}–{p_in_raw.p_in.max():.0f} Pa"
)

# ── Interpolate onto dome-closed base timestamps ──────────────────────────────
# Re-use t_base (inside-wind timestamps) and _to_seconds helper from earlier cells.
p_base = pd.DataFrame(index=base.index)
p_base["dome_closed"] = base["dome_closed"]

t_pout = _to_seconds(p_out_raw.index)
t_pin = _to_seconds(p_in_raw.index)

p_base["p_out"] = np.interp(t_base, t_pout, p_out_raw["p_out"].astype(float).values)
p_base["p_in"] = np.interp(t_base, t_pin, p_in_raw["p_in"].astype(float).values)

# ── Raw and dynamic differential ─────────────────────────────────────────────
p_base["dP_raw"] = p_base["p_in"] - p_base["p_out"]

# 1-hour rolling median removes static height/calibration offset
p_base["dP_offset"] = (
    p_base["dP_raw"].rolling("1h", center=True, min_periods=60).median()
)
p_base["dP_dyn"] = p_base["dP_raw"] - p_base["dP_offset"]

# ── Air density from outside pressure (ideal gas) ──────────────────────────
p_base["rho"] = p_base["p_out"] / (R_air * T_assumed_K)

# ── Orifice flow rate ─────────────────────────────────────────────────────────
# Q > 0: inflow (inside pressure lower than outside); Q < 0: outflow
p_base["Q_m3s"] = (
    np.sign(-p_base["dP_dyn"])
    * C_d
    * A_gap
    * np.sqrt(np.maximum(2.0 * p_base["dP_dyn"].abs() / p_base["rho"], 0.0))
)

# Restrict to dome-closed, outside wind > 2 m/s (same filter as efficiency plot)
p_closed = p_base.loc[base["dome_closed"]].copy()
p_closed["outside_speed"] = base.loc[base["dome_closed"], "outside_speed"]
p_closed = p_closed[p_closed["outside_speed"] > 2.0]

roll_Q = p_closed["Q_m3s"].rolling("30min", center=True, min_periods=10).median()

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(
    p_closed.index,
    p_closed["p_out"],
    lw=0.5,
    alpha=0.6,
    color="steelblue",
    label="Outside (ESS:301)",
)
axes[0].plot(
    p_closed.index,
    p_closed["p_in"],
    lw=0.5,
    alpha=0.6,
    color="darkorange",
    label="Inside (ESS:113)",
)
axes[0].set_ylabel("Pressure (Pa)")
axes[0].legend(loc="upper right", fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    p_closed.index,
    p_closed["dP_dyn"],
    lw=0.4,
    alpha=0.5,
    color="gray",
    label="Dynamic ΔP (raw)",
)
axes[1].axhline(0, color="k", lw=0.8, ls="--")
axes[1].set_ylabel("ΔP$_{dyn}$ (Pa)\n(inside − outside)")
axes[1].grid(True, alpha=0.3)

axes[2].plot(
    roll_Q.index,
    roll_Q.values,
    lw=1.2,
    color="firebrick",
    label="30-min rolling median Q",
)
axes[2].fill_between(
    roll_Q.index,
    roll_Q.values,
    0,
    where=roll_Q.values > 0,
    alpha=0.2,
    color="steelblue",
    label="Inflow",
)
axes[2].fill_between(
    roll_Q.index,
    roll_Q.values,
    0,
    where=roll_Q.values < 0,
    alpha=0.2,
    color="darkorange",
    label="Outflow",
)
axes[2].axhline(0, color="k", lw=0.8, ls="--")
axes[2].set_ylabel(f"Flow rate Q (m³/s)\n$C_d$={C_d}, $A_{{gap}}$={A_gap:.1f} m²")
axes[2].set_xlabel("UTC")
axes[2].legend(loc="upper right", fontsize=8)
axes[2].grid(True, alpha=0.3)

fig.suptitle(
    f"Azimuth Gap Air Leak Rate (dome closed, outside > 2 m/s)\n"
    f"{t_start.iso[:10]} – {t_end.iso[:10]}  |  gap = π×{dome_diameter_m}m×{gap_height_m}m",
    fontsize=11,
)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

q_abs = p_closed["Q_m3s"].abs()
print(f"Median |Q|  : {q_abs.median():.1f} m³/s")
print(f"95th pct |Q|: {q_abs.quantile(0.95):.1f} m³/s")
print(f"Max |Q|     : {q_abs.max():.1f} m³/s")

## Air Leak Rate Through Azimuth Perimeter Gap

Infiltration via the orifice equation:

$$Q = C_d \cdot A_{gap} \cdot \sqrt{\frac{2\,|\Delta P_{dyn}|}{\rho}}$$

- $\Delta P_{dyn}$ = dynamic (wind-driven) pressure difference, obtained by subtracting a 1-hour rolling median from the raw inside−outside differential to remove the static sensor offset.
- $\rho$ = local air density from the outside pressure sensor.
- $C_d$ = 0.6 (sharp-edged gap).
- $A_{gap}$ = π × 30 m × 0.05 m ≈ 4.71 m² (full azimuth perimeter).

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
pressure_topic = "lsst.sal.ESS.pressure"
pressure_field = "pressureItem0"
outside_pres_index = 301  # ESS:301 outside weather station
inside_pres_index = 113  # ESS:113 inside dome

dome_diameter_m = 30.0  # outer dome diameter [m]
gap_height_m = 0.05  # azimuth perimeter gap height [m]
C_d = 0.6  # discharge coefficient (sharp-edged orifice)
R_air = 287.1  # specific gas constant for dry air [J/(kg·K)]
T_assumed_K = 283.0  # ~10 °C fallback if no temperature sensor

# Derived gap area
A_gap = np.pi * dome_diameter_m * gap_height_m
print(f"Gap area: π × {dome_diameter_m} m × {gap_height_m} m = {A_gap:.2f} m²")

# ── Fetch pressure series ────────────────────────────────────────────────────
p_out_raw = await client.select_time_series(
    pressure_topic,
    fields=[pressure_field],
    start=t_start,
    end=t_end,
    index=outside_pres_index,
)
p_in_raw = await client.select_time_series(
    pressure_topic,
    fields=[pressure_field],
    start=t_start,
    end=t_end,
    index=inside_pres_index,
)
if p_out_raw.empty or p_in_raw.empty:
    raise RuntimeError("Pressure data missing for one or both sensors.")

p_out_raw = p_out_raw[[pressure_field]].rename(columns={pressure_field: "p_out"})
p_in_raw = p_in_raw[[pressure_field]].rename(columns={pressure_field: "p_in"})
print(
    f"Outside pressure rows : {len(p_out_raw)}  range {p_out_raw.p_out.min():.0f}–{p_out_raw.p_out.max():.0f} Pa"
)
print(
    f"Inside  pressure rows : {len(p_in_raw)}  range {p_in_raw.p_in.min():.0f}–{p_in_raw.p_in.max():.0f} Pa"
)

# ── Interpolate onto dome-closed base timestamps ──────────────────────────────
# Re-use t_base (inside-wind timestamps) and _to_seconds helper from earlier cells.
p_base = pd.DataFrame(index=base.index)
p_base["dome_closed"] = base["dome_closed"]

t_pout = _to_seconds(p_out_raw.index)
t_pin = _to_seconds(p_in_raw.index)

p_base["p_out"] = np.interp(t_base, t_pout, p_out_raw["p_out"].astype(float).values)
p_base["p_in"] = np.interp(t_base, t_pin, p_in_raw["p_in"].astype(float).values)

# ── Raw and dynamic differential ─────────────────────────────────────────────
p_base["dP_raw"] = p_base["p_in"] - p_base["p_out"]

# 1-hour rolling median removes static height/calibration offset
p_base["dP_offset"] = (
    p_base["dP_raw"].rolling("1h", center=True, min_periods=60).median()
)
p_base["dP_dyn"] = p_base["dP_raw"] - p_base["dP_offset"]

# ── Air density from outside pressure (ideal gas) ──────────────────────────
p_base["rho"] = p_base["p_out"] / (R_air * T_assumed_K)

# ── Orifice flow rate ─────────────────────────────────────────────────────────
# Q > 0: inflow (inside pressure lower than outside); Q < 0: outflow
p_base["Q_m3s"] = (
    np.sign(-p_base["dP_dyn"])
    * C_d
    * A_gap
    * np.sqrt(np.maximum(2.0 * p_base["dP_dyn"].abs() / p_base["rho"], 0.0))
)

# Restrict to dome-closed, outside wind > 2 m/s (same filter as efficiency plot)
p_closed = p_base.loc[base["dome_closed"]].copy()
p_closed["outside_speed"] = base.loc[base["dome_closed"], "outside_speed"]
p_closed = p_closed[p_closed["outside_speed"] > 2.0]

roll_Q = p_closed["Q_m3s"].rolling("30min", center=True, min_periods=10).median()

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(
    p_closed.index,
    p_closed["p_out"],
    lw=0.5,
    alpha=0.6,
    color="steelblue",
    label="Outside (ESS:301)",
)
axes[0].plot(
    p_closed.index,
    p_closed["p_in"],
    lw=0.5,
    alpha=0.6,
    color="darkorange",
    label="Inside (ESS:113)",
)
axes[0].set_ylabel("Pressure (Pa)")
axes[0].legend(loc="upper right", fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    p_closed.index,
    p_closed["dP_dyn"],
    lw=0.4,
    alpha=0.5,
    color="gray",
    label="Dynamic ΔP (raw)",
)
axes[1].axhline(0, color="k", lw=0.8, ls="--")
axes[1].set_ylabel("ΔP$_{dyn}$ (Pa)\n(inside − outside)")
axes[1].grid(True, alpha=0.3)

axes[2].plot(
    roll_Q.index,
    roll_Q.values,
    lw=1.2,
    color="firebrick",
    label="30-min rolling median Q",
)
axes[2].fill_between(
    roll_Q.index,
    roll_Q.values,
    0,
    where=roll_Q.values > 0,
    alpha=0.2,
    color="steelblue",
    label="Inflow",
)
axes[2].fill_between(
    roll_Q.index,
    roll_Q.values,
    0,
    where=roll_Q.values < 0,
    alpha=0.2,
    color="darkorange",
    label="Outflow",
)
axes[2].axhline(0, color="k", lw=0.8, ls="--")
axes[2].set_ylabel(f"Flow rate Q (m³/s)\n$C_d$={C_d}, $A_{{gap}}$={A_gap:.1f} m²")
axes[2].set_xlabel("UTC")
axes[2].legend(loc="upper right", fontsize=8)
axes[2].grid(True, alpha=0.3)

fig.suptitle(
    f"Azimuth Gap Air Leak Rate (dome closed, outside > 2 m/s)\n"
    f"{t_start.iso[:10]} – {t_end.iso[:10]}  |  gap = π×{dome_diameter_m}m×{gap_height_m}m",
    fontsize=11,
)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

q_abs = p_closed["Q_m3s"].abs()
print(f"Median |Q|  : {q_abs.median():.1f} m³/s")
print(f"95th pct |Q|: {q_abs.quantile(0.95):.1f} m³/s")
print(f"Max |Q|     : {q_abs.max():.1f} m³/s")